# 🍇 SUELO INTELIGENTE - Viabilidad para Cultivo de Vid
## Análisis de datos reales con sensor ST03 (Blvd. 2000, Tijuana B.C.)
Este notebook está preparado para ejecutarse de forma directa y autónoma en **Google Colab** utilizando los datos reales de tu bitácora de campo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# 1. Carga del Dataset Real (Bitácora de Recolección de Emili J. Armenta)
datos_medidor = {
    'Fertilidad_uS_cm': [914, 209, 233, 202, 227, 218, 133, 337, 45, 27, 233, 115, 76, 55, 290, 103, 58, 14, 293, 126, 25, 88, 223, 41, 201, 24, 26, 31, 21, 52, 221, 292, 60, 21, 22, 13, 12, 20],
    'Humedad_Suelo_pct': [60, 66, 63, 59, 51, 57, 50, 55, 51, 52, 53, 49, 53, 16, 66, 68, 53, 60, 70, 64, 51, 50, 56, 63, 54, 31, 60, 55, 46, 52, 88, 86, 56, 46, 48, 16, 10, 50],
    'pH': [7.5, 7.0, 6.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.0, 7.0, 7.5, 7.5, 7.0, 7.0, 7.5, 7.5, 7.5, 7.0, 7.5, 7.5, 7.0, 7.5, 7.5, 7.5, 6.0, 6.5, 7.0, 7.5, 7.5, 7.5, 7.5, 7.5],
    'Temperatura_C': [20.9, 30.0, 36.2, 26.1, 21.5, 35.6, 32.1, 26.8, 23.3, 38.1, 27.5, 26.9, 25.2, 36.0, 30.6, 26.8, 22.7, 25.3, 28.5, 24.4, 21.9, 23.3, 24.9, 21.5, 18.9, 24.6, 21.6, 21.5, 27.9, 27.8, 26.8, 23.7, 18.0, 28.6, 26.6, 20.7, 15.0, 35.5],
    'Luz_Solar_LUX': [8865, 90700, 6500, 1, 1461, 36300, 1127, 1, 3700, 47800, 1223, 3, 24050, 46100, 8047, 4, 1760, 19556, 1458, 3, 9022, 15725, 9090, 2, 8677, 21500, 1090, 1, 17910, 26425, 1180, 3, 1980, 16067, 1210, 2, 1341, 47700],
    'Humedad_Amb_pct': [52, 35, 38, 54, 54, 40, 42, 55, 53, 39, 48, 54, 53, 49, 44, 54, 56, 54, 48, 53, 53, 52, 46, 56, 53, 47, 51, 51, 43, 40, 41, 51, 53, 39, 43, 53, 54, 39]
}

df = pd.DataFrame(datos_medidor)
print(f"Dataset cargado correctamente. Total de muestras reales: {len(df)}")

In [ ]:
# 2. Cálculo del Punto de Rocío (Dew Point) y Etiquetado de Viabilidad
def calcular_punto_rocio(temp, hum):
    a = 17.27
    b = 237.7
    alpha = ((a * temp) / (b + temp)) + np.log(hum/100.0)
    return (b * alpha) / (a - alpha)

df['Punto_Rocio_C'] = df.apply(lambda row: calcular_punto_rocio(row['Temperatura_C'], row['Humedad_Amb_pct']), axis=1)

def evaluar_vid(fila):
    ph_ok = 6.0 <= fila['pH'] <= 7.5
    temp_ok = 15.0 <= fila['Temperatura_C'] <= 25.0
    humedad_amb_baja = fila['Humedad_Amb_pct'] < 50
    buen_drenaje = fila['Humedad_Suelo_pct'] < 40
    buena_luz = fila['Luz_Solar_LUX'] > 50000

    condiciones_cumplidas = sum([ph_ok, temp_ok, humedad_amb_baja, buen_drenaje, buena_luz])
    return 1 if condiciones_cumplidas >= 3 else 0

df['Viabilidad_Vid'] = df.apply(evaluar_vid, axis=1)
df.head()

## 📈 Visualizaciones Solicitadas

In [ ]:
# Gráfico 1: Análisis de Parcela (Zona de Viabilidad para la Vid)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='pH', y='Humedad_Suelo_pct',
                hue='Viabilidad_Vid', size='Luz_Solar_LUX', sizes=(20, 250),
                palette={0: '#e74c3c', 1: '#2ecc71'}, alpha=0.8)
plt.axvline(x=6.0, color='gray', linestyle='--', label='Min pH (6.0)')
plt.axvline(x=7.5, color='gray', linestyle='--', label='Max pH (7.5)')
plt.axhline(y=40, color='blue', linestyle='--', label='Max Humedad (<40%)')
plt.title('Gráfico 1: Zona de Viabilidad para la Vid (Datos Reales)', fontsize=13, fontweight='bold')
plt.xlabel('Nivel de pH (Óptimo entre 6.0 y 7.5)', fontsize=11)
plt.ylabel('Humedad del Suelo % (Óptimo < 40%)', fontsize=11)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Matriz de Correlación de Pearson (Heatmap)
plt.figure(figsize=(12, 8))
correlaciones = df.corr()
sns.heatmap(correlaciones, annot=True, cmap='RdYlGn', fmt=".2f", linewidths=0.5)
plt.title('Gráfico 2: Matriz de Correlación de Variables del Suelo vs Viabilidad', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 3: Análisis Multivariante (PairPlot - Grid 7x7 con 49 subgráficos)
sns.set_theme(style="ticks")
pair_plot = sns.pairplot(df, hue='Viabilidad_Vid', palette={0: '#e74c3c', 1: '#2ecc71'}, 
                         diag_kind='kde', plot_kws={'alpha': 0.6})
pair_plot.fig.suptitle('Gráfico 3: Análisis de Interdependencia Multivariante (PairPlot)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

## 🤖 Entrenamiento del Modelo de Clasificación KNN

In [ ]:
X = df.drop('Viabilidad_Vid', axis=1)
y = df['Viabilidad_Vid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

accuracy = knn.score(X_test_scaled, y_test) * 100
print(f"Modelo K-Neighbors Classifier entrenado de manera exitosa.")
print(f"Precisión en datos de prueba: {accuracy:.2f}%")